# Чекпоинт 6

## Постановка задачи:

1. Реализовать модель [PAWN](https://www.sciencedirect.com/science/article/pii/S156625352500538X?ref=pdf_download&fr=RR-2&rr=9f97f735c8398b88)
2. Добавить в архитектуру PAWN дополнительные признаки на основе второй модели.

In [1]:
# Импорты

import polars as pl

## 1. Датасет

В качестве датасета для всех экспериментов используется [MAGE](https://huggingface.co/datasets/yaful/MAGE/viewer/default/train?row=18):

- Paper: https://arxiv.org/pdf/2305.13242

- GitHub: https://github.com/yafuly/MAGE?tab=readme-ov-file#-dataset

- Hugging Face: https://huggingface.co/datasets/yaful/MAGE/viewer/default/train?row=18

Датасет содержит большое количество доменов и моделей, что позволяет полноценно оченить работу детектора.

Более того, этот датасет используется в оригинальной статье [PAWN](https://www.sciencedirect.com/science/article/pii/S156625352500538X?ref=pdf_download&fr=RR-2&rr=9f97f735c8398b88)

Изначально в датасете использовалась следующее распределение по категориям: 

1 - Human-written, 0 - Machine-generated

Чтобы соответствовать оригинальной работе, категории были поменяны местами: 

0 - Human-written, 1 - Machine-generated

In [9]:
df_mage_train = pl.read_csv("checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/train.csv")
df_mage_valid = pl.read_csv("checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/valid.csv")
df_mage_test = pl.read_csv("checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/test.csv")

In [16]:
print(f"Кол-во наблюдений в трейне: {df_mage_train.height}")
print(f"Кол-во наблюдений в валидации: {df_mage_valid.height}")
print(f"Кол-во наблюдений в тесет: {df_mage_test.height}")

Кол-во наблюдений в трейне: 319071
Кол-во наблюдений в валидации: 56792
Кол-во наблюдений в тесет: 60743


In [13]:
df_mage_train.head()

text,label
str,i64
"""White girls very rarely date A…",0
"""I am a 23 year old male Indian…",0
"""Take three people, Persons A, …",0
"""(A) Work part-time in high sch…",0
"""When police introduce a new fo…",0


In [8]:
# Соотношение классов в трейне
df_mage_train.group_by("label").agg(pl.len().alias("count"))

label,count
i64,u32
1,225753
0,93318


In [10]:
# Соотношение классов в валидации
df_mage_valid.group_by("label").agg(pl.len().alias("count"))

label,count
i64,u32
1,27993
0,28799


In [11]:
# Соотношение классов в тесте
df_mage_test.group_by("label").agg(pl.len().alias("count"))

label,count
i64,u32
0,30265
1,30478


Для проведения большинства экспериментов с добавлением новых признаков использовалась сбалансированная подвыборка из трейна и валидации на 5000 и 2000 наблюдений, соответственно. Тестовая выборка использовалась полностью для замера финальных метрик.

Команда для генерации сэмплированных выборок:

```bash
uv run checkpoint_6/dataset/sample_dataset.py \
--train_dataset=checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/train.csv \
--valid_dataset=checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/valid.csv
```

In [17]:
df_mage_train_sampled = pl.read_csv("checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/train_sampled.csv")
df_mage_valid_sampled = pl.read_csv("checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/valid_sampled.csv")

In [19]:
# Соотношение классов в трейне
df_mage_train_sampled.group_by("label").agg(pl.len().alias("count"))

label,count
i64,u32
1,2500
0,2500


In [20]:
# Соотношение классов в валидации
df_mage_valid_sampled.group_by("label").agg(pl.len().alias("count"))

label,count
i64,u32
0,1000
1,1000


## Sanity check

Перед проведением экспериментов мы проверили качество базовой реализации PAWN-модели, чтобы сравнить результаты с оригинальной статьей.

![pawn_metrics.png](checkpoint_6/original_pawn_metrics/pawn_metrics.png)

### Setup


Для обучения использовался весь трейн датасет и замороженная LLM ```meta-llama/Llama-3.2-1B-Instruct```

Гиперпараметры для обучения идентичны оригинальной PAWN модели:

```yaml
model:
  primary_model_name: meta-llama/Llama-3.2-1B-Instruct
  max_length: 512
  metric_features: 256
  gates: 256
  mlp_hidden_features: 256
  mlp_hidden_layers: 3
  mlp_dropout: 0.0
  token_dropout: 0.15
  residual: true
  primary_model_metrics:
    - entropy
    - max_log_probs
    - next_token_log_probs
    - rank
    - top_p

optimizer:
  learning_rate: 1.0e-3
  weight_decay: 1.0e-2
  max_grad_norm: 1.0
  gradient_accumulation_steps: 1
  label_smoothing: 0.2
  pos_weight: 0.413

trainer:
  device: null
  seed: 42
  epochs: 5

data:
  batch_size: 64
  eval_batch_size: 64
```

Команда для запуска обучения: 

```bash
python checkpoint_6/train.py \
--config checkpoint_6/experiments/mage_llama_base_full.yaml \
--train_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/train.csv \
--valid_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/valid.csv \
--test_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/test.csv \
--output_dir checkpoint_6/experiments/mage_llama_base_full
```

In [5]:
path = "checkpoint_6/experiments/mage_llama_base_full/test_metrics.json"

full_pawn_results = (
    pl.read_json(path)
    .transpose(
        include_header=True,
        header_name="metrics",
    )
    .rename(
        {"column_0": "original"}
    )
)

full_pawn_results.show(limit=None)

metrics,original
str,f64
"""test_loss""",0.265103
"""test_accuracy""",0.927366
"""test_ai_f1""",0.925953
"""test_ai_precision""",0.947777
"""test_ai_recall""",0.905112
"""test_human_f1""",0.928726
"""test_human_precision""",0.908588
"""test_human_recall""",0.949777
"""test_roc_auc""",0.975026


# Эксперименты

Все эксперименты запущены на сэмплированной трейн выборке из 5000 наблюдений

## 1. Бейзлайн

Результаты базовой PAWN модели на семплированной выборке.

Гиперпараметры: 

```yaml
model:
  primary_model_name: meta-llama/Llama-3.2-1B-Instruct
  max_length: 512
  metric_features: 256
  gates: 256
  mlp_hidden_features: 256
  mlp_hidden_layers: 3
  mlp_dropout: 0.0
  token_dropout: 0.15
  residual: true
  primary_model_metrics:
    - entropy
    - max_log_probs
    - next_token_log_probs
    - rank
    - top_p

optimizer:
  learning_rate: 1.0e-3
  weight_decay: 1.0e-2
  max_grad_norm: 1.0
  gradient_accumulation_steps: 1

trainer:
  device: null
  seed: 42
  epochs: 5

data:
  batch_size: 64
  eval_batch_size: 64
```

Команда для запуска обучения: 

```bash
python checkpoint_6/train.py \
--config checkpoint_6/experiments/mage_llama_base.yaml \
--train_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/train_sampled.csv \
--valid_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/valid_sampled.csv \
--test_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/test.csv \
--output_dir checkpoint_6/experiments/mage_llama_base
```

In [9]:
path = "checkpoint_6/experiments/mage_llama_base/test_metrics.json"

results = (
    pl.read_json(path)
    .transpose(
        include_header=True,
        header_name="metrics",
    )
    .rename(
        {"column_0": "baseline_pawn"}
    )
)

results.show(limit=None)

metrics,baseline_pawn
str,f64
"""test_loss""",0.489551
"""test_accuracy""",0.804488
"""test_human_f1""",0.796727
"""test_human_precision""",0.826521
"""test_human_recall""",0.769007
"""test_ai_f1""",0.811677
"""test_ai_precision""",0.785447
"""test_ai_recall""",0.83972
"""test_roc_auc""",0.887705


## 2. PAWN + xxpl

Результаты PAWN модели с добавлением [xxpl](https://arxiv.org/abs/2401.12070) к базовым метрикам.

Гиперпараметры: 

```yaml
model:
  primary_model_name: meta-llama/Llama-3.2-1B-Instruct
  second_model_name: meta-llama/Llama-3.2-1B
  max_length: 512
  metric_features: 256
  gates: 256
  mlp_hidden_features: 256
  mlp_hidden_layers: 3
  mlp_dropout: 0.0
  token_dropout: 0.15
  residual: true
  primary_model_metrics:
    - entropy
    - max_log_probs
    - next_token_log_probs
    - rank
    - top_p
  return_xppl: true

optimizer:
  learning_rate: 1.0e-3
  weight_decay: 1.0e-2
  max_grad_norm: 1.0
  gradient_accumulation_steps: 1

trainer:
  device: null
  seed: 42
  epochs: 5

data:
  batch_size: 32
  eval_batch_size: 32
```

Команда для запуска обучения: 

```bash
python checkpoint_6/train.py \
--config checkpoint_6/experiments/mage_llama_xppl.yaml \
--train_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/train_sampled.csv \
--valid_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/valid_sampled.csv \
--test_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/test.csv \
--output_dir checkpoint_6/experiments/mage_llama_xppl
```

In [ ]:
path = "checkpoint_6/experiments/mage_llama_xppl/test_metrics.json"

results_xppl = (
    pl.read_json(path)
    .transpose(
        include_header=True,
        header_name="metrics",
    )
    .rename(
        {"column_0": "xppl"}
    )
)

results = results.join(results_xppl, on="metrics")

results.show(limit=None)

metrics,baseline_pawn,xppl
str,f64,f64
"""test_loss""",0.489551,0.561837
"""test_accuracy""",0.804488,0.784782
"""test_human_f1""",0.796727,0.770105
"""test_human_precision""",0.826521,0.823158
"""test_human_recall""",0.769007,0.723476
"""test_ai_f1""",0.811677,0.797697
"""test_ai_precision""",0.785447,0.754884
"""test_ai_recall""",0.83972,0.845659
"""test_roc_auc""",0.887705,0.86759


## 3. PAWN + DFT

Результаты PAWN модели с добавлением [DFT](https://arxiv.org/pdf/2508.11343) к базовым метрикам.

Гиперпараметры: 

```yaml
model:
  primary_model_name: meta-llama/Llama-3.2-1B-Instruct
  max_length: 512
  metric_features: 256
  gates: 256
  mlp_hidden_features: 256
  mlp_hidden_layers: 3
  mlp_dropout: 0.0
  token_dropout: 0.15
  residual: true
  primary_model_metrics:
    - entropy
    - max_log_probs
    - next_token_log_probs
    - rank
    - top_p
    - fft

optimizer:
  learning_rate: 1.0e-3
  weight_decay: 1.0e-2
  max_grad_norm: 1.0
  gradient_accumulation_steps: 1

trainer:
  device: null
  seed: 42
  epochs: 5

data:
  batch_size: 64
  eval_batch_size: 64
```

Команда для запуска обучения: 

```bash
python checkpoint_6/train.py \
--config checkpoint_6/experiments/mage_llama_fft.yaml \
--train_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/train_sampled.csv \
--valid_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/valid_sampled.csv \
--test_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/test.csv \
--output_dir checkpoint_6/experiments/mage_llama_fft
```

In [11]:
path = "checkpoint_6/experiments/mage_llama_fft/test_metrics.json"

results_fft = (
    pl.read_json(path)
    .transpose(
        include_header=True,
        header_name="metrics",
    )
    .rename(
        {"column_0": "fft"}
    )
)

results = results.join(results_fft, on="metrics")

results.show(limit=None)

metrics,baseline_pawn,xppl,fft
str,f64,f64,f64
"""test_loss""",0.489551,0.561837,0.640243
"""test_accuracy""",0.804488,0.784782,0.71549
"""test_human_f1""",0.796727,0.770105,0.685003
"""test_human_precision""",0.826521,0.823158,0.763893
"""test_human_recall""",0.769007,0.723476,0.620882
"""test_ai_f1""",0.811677,0.797697,0.740596
"""test_ai_precision""",0.785447,0.754884,0.682548
"""test_ai_recall""",0.83972,0.845659,0.809436
"""test_roc_auc""",0.887705,0.86759,0.803989


## 4. PAWN + Second model metrics + xppl

Результаты PAWN модели с добавлением метрик второй модели и xxpl к базовым метрикам.

Гиперпараметры: 

```yaml
model:
  primary_model_name: meta-llama/Llama-3.2-1B-Instruct
  second_model_name: meta-llama/Llama-3.2-1B
  max_length: 512
  metric_features: 256
  gates: 256
  mlp_hidden_features: 256
  mlp_hidden_layers: 3
  mlp_dropout: 0.0
  token_dropout: 0.15
  residual: true
  primary_model_metrics:
    - entropy
    - max_log_probs
    - next_token_log_probs
    - rank
    - top_p
  second_model_metrics:
    - entropy
    - max_log_probs
    - next_token_log_probs
    - rank
    - top_p
  return_xppl: true

optimizer:
  learning_rate: 1.0e-3
  weight_decay: 1.0e-2
  max_grad_norm: 1.0
  gradient_accumulation_steps: 1

trainer:
  device: null
  seed: 42
  epochs: 5

data:
  batch_size: 32
  eval_batch_size: 32
```

Команда для запуска обучения: 

```bash
python checkpoint_6/train.py \
--config checkpoint_6/experiments/mage_llama_metrics_xppl.yaml \
--train_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/train_sampled.csv \
--valid_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/valid_sampled.csv \
--test_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/test.csv \
--output_dir checkpoint_6/experiments/mage_llama_metrics_xppl
```

In [12]:
path = "checkpoint_6/experiments/mage_llama_metrics_xppl/test_metrics.json"

results_metrics_xppl = (
    pl.read_json(path)
    .transpose(
        include_header=True,
        header_name="metrics",
    )
    .rename(
        {"column_0": "metrics_xppl"}
    )
)

results = results.join(results_metrics_xppl, on="metrics")

results.show(limit=None)

metrics,baseline_pawn,xppl,fft,metrics_xppl
str,f64,f64,f64,f64
"""test_loss""",0.489551,0.561837,0.640243,0.401655
"""test_accuracy""",0.804488,0.784782,0.71549,0.836821
"""test_human_f1""",0.796727,0.770105,0.685003,0.838383
"""test_human_precision""",0.826521,0.823158,0.763893,0.827587
"""test_human_recall""",0.769007,0.723476,0.620882,0.849463
"""test_ai_f1""",0.811677,0.797697,0.740596,0.835228
"""test_ai_precision""",0.785447,0.754884,0.682548,0.846486
"""test_ai_recall""",0.83972,0.845659,0.809436,0.824267
"""test_roc_auc""",0.887705,0.86759,0.803989,0.915595


## 5. PAWN + Second model metrics + xppl + Second model hidden states

Результаты PAWN модели с добавлением метрик второй модели, xxpl и hidden states второй модели.

hidden states второй модели конкатенировались с hidden states первой модели при обучении.

Гиперпараметры: 

```yaml
model:
  primary_model_name: meta-llama/Llama-3.2-1B-Instruct
  second_model_name: meta-llama/Llama-3.2-1B
  max_length: 512
  metric_features: 256
  gates: 256
  mlp_hidden_features: 256
  mlp_hidden_layers: 3
  mlp_dropout: 0.0
  token_dropout: 0.15
  residual: true
  primary_model_metrics:
    - entropy
    - max_log_probs
    - next_token_log_probs
    - rank
    - top_p
  second_model_metrics:
    - entropy
    - max_log_probs
    - next_token_log_probs
    - rank
    - top_p
  return_xppl: true
  return_second_model_hs: true

optimizer:
  learning_rate: 1.0e-3
  weight_decay: 1.0e-2
  max_grad_norm: 1.0
  gradient_accumulation_steps: 1

trainer:
  device: null
  seed: 42
  epochs: 5

data:
  batch_size: 32
  eval_batch_size: 32
```

Команда для запуска обучения: 

```bash
python checkpoint_6/train.py \
--config checkpoint_6/experiments/mage_llama_metrics_xppl_hs.yaml \
--train_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/train_sampled.csv \
--valid_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/valid_sampled.csv \
--test_dataset checkpoint_6/dataset/MAGE/testbeds/cross_domains_cross_models/test.csv \
--output_dir checkpoint_6/experiments/mage_llama_metrics_xppl_hs
```

In [13]:
path = "checkpoint_6/experiments/mage_llama_metrics_xppl_hs/test_metrics.json"

results_metrics_xppl_hs = (
    pl.read_json(path)
    .transpose(
        include_header=True,
        header_name="metrics",
    )
    .rename(
        {"column_0": "metrics_xppl_hs"}
    )
)

results = results.join(results_metrics_xppl_hs, on="metrics")

results.show(limit=None)

metrics,baseline_pawn,xppl,fft,metrics_xppl,metrics_xppl_hs
str,f64,f64,f64,f64,f64
"""test_loss""",0.489551,0.561837,0.640243,0.401655,0.507701
"""test_accuracy""",0.804488,0.784782,0.71549,0.836821,0.784864
"""test_human_f1""",0.796727,0.770105,0.685003,0.838383,0.773636
"""test_human_precision""",0.826521,0.823158,0.763893,0.827587,0.813071
"""test_human_recall""",0.769007,0.723476,0.620882,0.849463,0.737849
"""test_ai_f1""",0.811677,0.797697,0.740596,0.835228,0.795031
"""test_ai_precision""",0.785447,0.754884,0.682548,0.846486,0.761584
"""test_ai_recall""",0.83972,0.845659,0.809436,0.824267,0.831551
"""test_roc_auc""",0.887705,0.86759,0.803989,0.915595,0.865493
